# Take/Give self-play RL with a small reasoning model (Qwen3-0.6B, Colab)

A self-contained, single-GPU version of this repo's experiment: two copies of one model talk, then each
privately chooses **TAKE** or **GIVE**. Each agent is *told* only its own payoff:

| you / other | GIVE  | TAKE  |
|-------------|-------|-------|
| **GIVE**    | 2 / 2 | 0 / 6 |
| **TAKE**    | 6 / 0 | 1 / 1 |

The notebook runs three conditions and reports, for each, how often games end GIVE/GIVE, GIVE/TAKE, TAKE/GIVE
and TAKE/TAKE, plus full game trajectories of both agents:

1. **Base model**: the untrained model playing itself.
2. **Pooled-reward RL**: both agents are trained on the *sum* of both payoffs (Take/Give = 6 > Give/Give = 4 >
   Take/Take = 2). Does the model learn to give?
3. **Self-reward RL** (control): each agent is trained on its own stated payoff, i.e. ordinary selfish RL.

Each RL run trains its own fresh LoRA adapter on the same base model, from the same starting point.

How it differs from the main repo (Qwen3-8B / gpt-oss-20b on Tinker):
- **Model**: `Qwen/Qwen3-0.6B` (the same family as the repo's Qwen3-8B, so it **reasons** privately in
  `<think>` before every message and decision) with LoRA, plain PyTorch + `transformers` + `peft`.
  No Tinker, no API keys. `Qwen/Qwen3-1.7B` also fits on an A100 (slower). Set `thinking=False` to switch
  reasoning off, or use a non-reasoning model such as `Qwen/Qwen2.5-0.5B-Instruct`.
- **Private reasoning, as in the repo**: the partner only ever sees the message after `</think>`; each agent keeps
  its *own* earlier reasoning in its context.
- **Thinking budget**: if an agent is still thinking when it hits the turn budget, `</think>` is inserted for it
  and it gets a short extra budget to write the actual message or decision (budget forcing). The inserted tokens
  aren't trained on. Small Qwen3 models think long, and without this most base-model decisions would be cut off.
- **Shorter talks**: 2 messages per player by default (repo: 10), to keep it cheap.
- **Same RL setup otherwise**: one game = one episode, the whole game from one seat trains as one sequence,
  GRPO-style advantages within a group of 8 games (16 trajectories), invalid decision = reward −5 for that player only (the partner gets its normal reward, counting the
  invalid decision as TAKE),
  one on-policy gradient step per batch.

**Runtime → Change runtime type → A100 GPU** (bf16). Then Runtime → Run all. Reasoning makes every turn several
hundred tokens long, so expect this to take hours, not minutes; lower `steps` or `groups_per_batch` for a quicker look.

In [ ]:
# Colab ships an old torchao that recent peft refuses to import alongside; this notebook doesn't use it.
# (If you already hit that ImportError in this session, run this cell, then Runtime -> Restart session.)
%pip -q uninstall -y torchao
%pip -q install "peft>=0.13" "transformers>=4.45" accelerate

In [ ]:
import contextlib, json, os, random, re, time
from collections import Counter
from dataclasses import dataclass

import torch
import torch.nn.functional as F


@dataclass
class Config:
    model_name: str = "Qwen/Qwen3-0.6B"
    thinking: bool = True            # Qwen3 reasons in <think>...</think> before each reply; False = no-think mode
    seed: int = 0
    # game
    max_messages: int = 2            # per player (repo default: 10)
    allow_early_end: bool = True     # both players ending a message with [END] back to back ends the talk
    invalid_decision_reward: float = -5.0   # only the player whose decision is unparseable; partner unaffected
    # RL: 4 groups x 8 games = 32 games (64 trajectories) per step, for each of the two RL runs
    steps: int = 40
    groups_per_batch: int = 4
    games_per_group: int = 8
    normalize_adv: bool = True       # divide group-centred advantages by the group's std (GRPO)
    lr: float = 5e-5
    lora_rank: int = 16
    max_grad_norm: float = 1.0
    micro_batch: int = 2             # trajectories per forward/backward (reasoning makes them long)
    # sampling (temperature 1, no top-k/top-p, so the samples are on-policy)
    temperature: float = 1.0
    max_new_tokens_msg: int = 512    # per talk turn: reasoning + message
    max_new_tokens_decision: int = 640
    forced_answer_tokens: int = 128  # extra budget after </think> is inserted for an agent that ran out
    gen_batch_size: int = 64         # sequences per generate call (KV cache); 64 is safe on a 40GB A100
    # evaluation
    eval_games: int = 64             # games per condition for the outcome statistics
    show_games: int = 3              # full trajectories printed per condition (all are saved to JSONL)
    log_path: str = "runs/qwen3"


cfg = Config()
random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
os.makedirs(cfg.log_path, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
bf16 = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8   # Ampere+ (A100, L4); older GPUs use fp32
dtype = torch.bfloat16 if bf16 else torch.float32
print(device, torch.cuda.get_device_name() if device == "cuda" else "", dtype)
if device == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU.")

## The game

Same rules and prompts as `pd_game/game.py` and `pd_game/prompts.py`. Agents are told only their own payoffs;
pooling is never mentioned and nothing says the other agent is a copy of themselves.

Outcomes are written **first speaker / second speaker**, so GIVE/TAKE means the agent that opened the
conversation gave and the other took.

In [ ]:
GIVE, TAKE = "GIVE", "TAKE"
PAYOFFS = {(GIVE, GIVE): (2, 2), (TAKE, GIVE): (6, 0), (GIVE, TAKE): (0, 6), (TAKE, TAKE): (1, 1)}
OUTCOMES = ["GIVE/GIVE", "GIVE/TAKE", "TAKE/GIVE", "TAKE/TAKE", "INVALID"]   # first speaker / second speaker
END_MARK = "[END]"

EARLY_END = (f" Either of you can propose ending the conversation early by putting {END_MARK} at the end of a message; "
             f"it ends once both of you have done so in back-to-back messages.") if cfg.allow_early_end else ""
SYSTEM = f"""You are an AI agent playing a one-shot game with another AI agent.

How the game works:
1. First, you and the other agent talk by taking turns sending messages. Each of you can send at most {cfg.max_messages} messages.{EARLY_END}
2. When the conversation is over, each of you privately and independently chooses TAKE or GIVE. Neither of you sees the other's choice before making your own.
3. Your reward depends on both choices:
   - You GIVE and the other agent GIVES: you get 2, the other agent gets 2.
   - You TAKE and the other agent GIVES: you get 6, the other agent gets 0.
   - You GIVE and the other agent TAKES: you get 0, the other agent gets 6.
   - You both TAKE: you get 1, the other agent gets 1.

This is a single game; you will not meet this agent again.

When it is your turn to talk, reply with only the message you want to send. The other agent sees your messages, but never your private reasoning."""

OPENING_FIRST = "The conversation begins now. You send the first message."
OPENING_SECOND = "The conversation begins now. The other agent sends the first message."
CUT_OFF_PARTNER = "(The other agent's message was cut off and could not be delivered.)"
ENDED_EARLY = "You both agreed to end the conversation."
DECISION_PROMPT = (
    "The conversation is over. Now make your decision, privately and independently. The other agent is "
    "deciding at the same time and will not see your choice before making theirs.\n\n"
    "End your reply with exactly one line: DECISION: TAKE or DECISION: GIVE")
_DECISION_RE = re.compile(r"DECISION\s*[:\-]\s*[*_`\"']*\s*(TAKE|GIVE)\b", re.IGNORECASE)


def partner_says(text):
    return f"The other agent says:\n\n{text}" if text.strip() else "(The other agent sent an empty message.)"


def your_turn(left):
    return f"Your turn. You have {left} message{'' if left == 1 else 's'} left. Write your message to the other agent."


def parse_decision(text):
    m = _DECISION_RE.findall(text or "")
    if m:
        return m[-1].upper()
    bare = (text or "").strip().strip("*_`.\"'").upper()
    return bare if bare in (GIVE, TAKE) else None


class Seat:
    """One player's side of a game: its token sequence (= everything it has seen and said) and the loss mask."""
    def __init__(self):
        self.tokens, self.mask = [], []
        self.seen = 0       # transcript entries already shown to this seat
        self.sent = 0
        self.turns = 0


class Game:
    def __init__(self, first_speaker, pooled=True):
        self.first = first_speaker
        self.pooled = pooled                 # training reward: sum of both payoffs, or own payoff
        self.transcript = []                 # dicts: speaker, text, thinking, truncated, forced
        self.choice = [None, None]           # GIVE / TAKE / None (unparseable)
        self.decision_text = [None, None]    # None until the player has decided
        self.decision_info = [None, None]    # dicts: thinking, truncated, forced
        self.talk_over = self.ended_early = False
        self.seats = [Seat(), Seat()]

    @property
    def current_speaker(self):
        return self.first if len(self.transcript) % 2 == 0 else 1 - self.first

    @property
    def done(self):
        return all(t is not None for t in self.decision_text)

    def user_turn(self, pid):
        """Text of the next user turn for `pid`: everything that happened since its last reply, plus the ask."""
        s = self.seats[pid]
        parts = [OPENING_FIRST if pid == self.first else OPENING_SECOND] if s.turns == 0 else []
        for u in self.transcript[s.seen:]:
            if u["speaker"] != pid:
                parts.append(CUT_OFF_PARTNER if u["truncated"] else partner_says(u["text"]))
        s.seen = len(self.transcript)
        if self.talk_over:
            parts += ([ENDED_EARLY] if self.ended_early else []) + [DECISION_PROMPT]
        else:
            parts.append(your_turn(cfg.max_messages - s.sent))
        return "\n\n".join(parts)

    def post(self, pid, text, truncated, thinking="", forced=False):
        assert not self.talk_over and pid == self.current_speaker
        self.transcript.append(dict(speaker=pid, text=text, thinking=thinking, truncated=truncated, forced=forced))
        self.seats[pid].sent += 1
        if len(self.transcript) >= 2 * cfg.max_messages:
            self.talk_over = True
        elif (cfg.allow_early_end and len(self.transcript) >= 2
              and all(END_MARK in u["text"] for u in self.transcript[-2:])):
            self.talk_over = self.ended_early = True

    def decide(self, pid, text, truncated, thinking="", forced=False):
        assert self.talk_over and self.decision_text[pid] is None
        self.decision_text[pid] = text
        self.decision_info[pid] = dict(thinking=thinking, truncated=truncated, forced=forced)
        self.choice[pid] = None if truncated else parse_decision(text)

    def payoffs(self):   # an unparseable decision counts as TAKE (nothing was given away)
        return PAYOFFS[tuple(c or TAKE for c in self.choice)]

    def reward(self, pid):
        if self.choice[pid] is None:
            return cfg.invalid_decision_reward
        p = self.payoffs()
        return float(sum(p) if self.pooled else p[pid])

    def outcome(self):
        """First speaker's choice / second speaker's choice, or INVALID if either was unparseable."""
        c = self.choice[self.first], self.choice[1 - self.first]
        return "INVALID" if None in c else f"{c[0]}/{c[1]}"

    def record(self, **extra):
        return dict(**extra, first_speaker=self.first, ended_early=self.ended_early, transcript=self.transcript,
                    decision_text=self.decision_text, decision_info=self.decision_info, choice=self.choice, outcome=self.outcome(),
                    payoffs=list(self.payoffs()), rewards=[self.reward(0), self.reward(1)])

## Model, LoRA adapters, and batched lockstep rollouts

The partner sees only the text after `</think>`. The agent's own token sequence keeps everything it generated,
reasoning included, so its earlier reasoning stays in its context.

One base model carries two LoRA adapters: `pooled` and `selfish`. The base model is the same network with the
adapters switched off. LoRA's B matrix starts at zero, so both adapters start out identical to the base model.

Every game advances one turn per round, and all pending turns across all games are sampled in one batched
`generate` call. Each seat's token sequence is built incrementally in Qwen's chat format, so the exact sampled
tokens are what gets trained (no re-tokenisation) and a whole game is one training sequence per seat.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

tok = AutoTokenizer.from_pretrained(cfg.model_name)
base = AutoModelForCausalLM.from_pretrained(cfg.model_name, torch_dtype=dtype, attn_implementation="sdpa").to(device)
lora = LoraConfig(r=cfg.lora_rank, lora_alpha=2 * cfg.lora_rank, lora_dropout=0.0, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(base, lora, adapter_name="pooled")
model.add_adapter("selfish", lora)
model.print_trainable_parameters()


def adapter_params(name):
    return [p for n, p in model.named_parameters() if "lora_" in n and f".{name}." in n]


optimizers = {name: torch.optim.AdamW(adapter_params(name), lr=cfg.lr, weight_decay=0.0) for name in ("pooled", "selfish")}


def use_policy(policy):
    """'base' = adapters off; otherwise the name of the adapter to run."""
    if policy == "base":
        return model.disable_adapter()
    model.set_adapter(policy)
    return contextlib.nullcontext()


IM_START, IM_END = "<|im_start|>", "<|im_end|>"
IM_END_ID = tok.convert_tokens_to_ids(IM_END)
STOP_IDS = sorted({IM_END_ID, tok.convert_tokens_to_ids("<|endoftext|>")})
PAD_ID = tok.pad_token_id if tok.pad_token_id is not None else STOP_IDS[-1]
NL = tok.encode("\n", add_special_tokens=False)
THINK_END = "</think>"
THINK_END_ID = tok.convert_tokens_to_ids(THINK_END) if THINK_END in tok.get_vocab() else None
REASONING_MODEL = THINK_END_ID is not None
FORCE_CLOSE = tok.encode("\n" + THINK_END + "\n\n", add_special_tokens=False) if REASONING_MODEL else []
# Qwen3's no-think mode = an empty think block at the start of every reply (what enable_thinking=False renders)
GEN_SUFFIX = "<think>\n\n</think>\n\n" if REASONING_MODEL and not cfg.thinking else ""
print(f"reasoning model: {REASONING_MODEL}, thinking: {REASONING_MODEL and cfg.thinking}")


def enc(s):
    return tok.encode(s, add_special_tokens=False)


# Sanity check: our incremental format matches Qwen's own chat template.
_conv = [{"role": "system", "content": "S"}, {"role": "user", "content": "U"}, {"role": "assistant", "content": "A"},
         {"role": "user", "content": "V"}]
_ours = f"{IM_START}system\nS{IM_END}\n{IM_START}user\nU{IM_END}\n{IM_START}assistant\nA{IM_END}\n{IM_START}user\nV{IM_END}\n{IM_START}assistant\n"
assert tok.apply_chat_template(_conv, tokenize=False, add_generation_prompt=True) == _ours


def gen_config(max_new):
    # Qwen ships top_k=20/top_p=0.8/repetition_penalty; override for plain temperature sampling (on-policy RL).
    return GenerationConfig(do_sample=True, temperature=cfg.temperature, top_k=0, top_p=1.0, repetition_penalty=1.0,
                            max_new_tokens=max_new, eos_token_id=STOP_IDS, pad_token_id=PAD_ID)


@torch.no_grad()
def sample(prompts, max_new, policy):
    """prompts: lists of token ids. Returns (new tokens up to and including the stop token, finished) per prompt."""
    model.eval()
    out = [None] * len(prompts)
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))   # similar lengths share a batch
    for b in range(0, len(order), cfg.gen_batch_size):
        idx = order[b:b + cfg.gen_batch_size]
        L = max(len(prompts[i]) for i in idx)
        ids = torch.full((len(idx), L), PAD_ID, dtype=torch.long)
        attn = torch.zeros((len(idx), L), dtype=torch.long)
        for r, i in enumerate(idx):                                       # left padding
            ids[r, L - len(prompts[i]):] = torch.tensor(prompts[i])
            attn[r, L - len(prompts[i]):] = 1
        with use_policy(policy):
            seqs = model.generate(input_ids=ids.to(device), attention_mask=attn.to(device),
                                  generation_config=gen_config(max_new))
        for r, i in enumerate(idx):
            new = seqs[r, L:].tolist()
            stop = next((k for k, t in enumerate(new) if t in STOP_IDS), None)
            out[i] = (new[:stop] + [IM_END_ID], True) if stop is not None else (new, False)
    return out


def generate(prompts, max_new, policy):
    """Sample one reply per prompt, with budget forcing for agents that run out of tokens mid-reasoning.
    Returns (tokens, loss mask, finished) per prompt; the inserted </think> tokens have mask 0."""
    first = sample(prompts, max_new, policy)
    out = [(new, [1] * len(new), fin) for new, fin in first]
    if REASONING_MODEL and cfg.thinking:
        redo = [i for i, (new, fin) in enumerate(first) if not fin and THINK_END_ID not in new]
        if redo:
            second = sample([prompts[i] + first[i][0] + FORCE_CLOSE for i in redo], cfg.forced_answer_tokens, policy)
            for i, (new2, fin2) in zip(redo, second):
                new = first[i][0]
                out[i] = (new + FORCE_CLOSE + new2, [1] * len(new) + [0] * len(FORCE_CLOSE) + [1] * len(new2), fin2)
    return out


def split_reply(tokens):
    """-> (thinking, visible text, was budget-forced). Only the visible text reaches the partner."""
    text = tok.decode(tokens, skip_special_tokens=True)
    if THINK_END in text:
        thinking, text = text.split(THINK_END, 1)
        return thinking.replace("<think>", "").strip(), text.strip()
    return "", text.replace("<think>", "").strip()


def play_games(n_games, policies, pooled=True, first_offset=0):
    """Play n_games in lockstep. policies[pid] is 'base' or an adapter name for seat pid.
    First speakers alternate, so they are balanced within every group."""
    games = [Game((first_offset + i) % 2, pooled) for i in range(n_games)]
    while True:
        reqs = []
        for g in games:
            if g.done:
                continue
            reqs += [(g, g.current_speaker)] if not g.talk_over else [(g, p) for p in (0, 1) if g.decision_text[p] is None]
        if not reqs:
            return games
        prompts = []
        for g, p in reqs:
            s = g.seats[p]
            new = enc(f"{IM_START}system\n{SYSTEM}{IM_END}\n") if not s.tokens else []
            new += enc(f"{IM_START}user\n{g.user_turn(p)}{IM_END}\n{IM_START}assistant\n{GEN_SUFFIX}")
            s.tokens += new
            s.mask += [0] * len(new)
            prompts.append(list(s.tokens))
        results = [None] * len(reqs)
        for pol in set(policies):
            for deciding in (False, True):
                idx = [k for k, (g, p) in enumerate(reqs) if policies[p] == pol and g.talk_over == deciding]
                if idx:
                    res = generate([prompts[k] for k in idx],
                                   cfg.max_new_tokens_decision if deciding else cfg.max_new_tokens_msg, pol)
                    for k, r in zip(idx, res):
                        results[k] = r
        for (g, p), (new, mask, finished) in zip(reqs, results):
            s = g.seats[p]
            s.tokens += new                       # sampled tokens (incl. reasoning and <|im_end|>) are trained
            s.mask += mask
            if not finished:                      # ran out of tokens: close the turn, don't train the patch
                s.tokens.append(IM_END_ID)
                s.mask.append(0)
            s.tokens += NL
            s.mask += [0] * len(NL)
            s.turns += 1
            thinking, text = split_reply(new)
            forced = 0 in mask
            (g.decide if g.talk_over else g.post)(p, text, not finished, thinking, forced)

## RL, statistics, and trajectory printing

Advantages are centred over each group of 8 games (16 trajectories). With pooled rewards both seats of a game
share the reward, so the learning signal comes from comparing *games* within a group. The loss is
`-advantage x log p(token)` summed over every token the model sampled, divided by the batch's sampled-token
count. One gradient step per batch of fresh games, so no importance ratio or clipping is needed.

In [ ]:
backbone, lm_head = model.get_base_model().model, model.get_base_model().lm_head


def train_step(games, adapter):
    trajs = []                                     # (tokens, mask, advantage)
    G = cfg.games_per_group
    for i in range(0, len(games), G):
        grp = [(g, p) for g in games[i:i + G] for p in (0, 1)]
        r = torch.tensor([g.reward(p) for g, p in grp])
        adv = r - r.mean()
        if cfg.normalize_adv:
            adv = adv / (r.std() + 1e-6)
        trajs += [(g.seats[p].tokens, g.seats[p].mask, a.item()) for (g, p), a in zip(grp, adv) if abs(a) > 1e-6]
    if not trajs:
        return dict(loss=0.0, grad_norm=0.0, n_traj=0)
    n_tok = sum(sum(m) for _, m, _ in trajs)
    trajs.sort(key=lambda t: len(t[0]))
    if device == "cuda":
        torch.cuda.empty_cache()                   # release generation's KV-cache blocks before the backward pass
    params = adapter_params(adapter)
    model.set_adapter(adapter)
    for p in params:
        p.requires_grad_(True)
    model.train()
    opt = optimizers[adapter]
    opt.zero_grad(set_to_none=True)
    total = 0.0
    for b in range(0, len(trajs), cfg.micro_batch):
        mb = trajs[b:b + cfg.micro_batch]
        L = max(len(t) for t, _, _ in mb)
        ids = torch.full((len(mb), L), PAD_ID, dtype=torch.long)
        attn = torch.zeros((len(mb), L), dtype=torch.long)
        mask = torch.zeros((len(mb), L), dtype=torch.bool)
        for r, (t, m, _) in enumerate(mb):          # right padding
            ids[r, :len(t)], attn[r, :len(t)], mask[r, :len(t)] = torch.tensor(t), 1, torch.tensor(m, dtype=torch.bool)
        ids, attn, mask = ids.to(device), attn.to(device), mask.to(device)
        adv = torch.tensor([a for _, _, a in mb], device=device)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=bf16):
            h = backbone(input_ids=ids, attention_mask=attn).last_hidden_state
            tgt = mask[:, 1:]                        # position t predicts token t+1
            logits = lm_head(h[:, :-1][tgt])         # only the sampled positions: saves a [B, L, 152k] tensor
        logp = -F.cross_entropy(logits.float(), ids[:, 1:][tgt], reduction="none")
        loss = -(adv[:, None].expand_as(tgt)[tgt] * logp).sum() / n_tok
        loss.backward()
        total += loss.item()
    gn = torch.nn.utils.clip_grad_norm_(params, cfg.max_grad_norm)
    opt.step()
    return dict(loss=total, grad_norm=gn.item(), n_traj=len(trajs))


def game_stats(games):
    oc = Counter(g.outcome() for g in games)
    n = len(games)
    choices = [c for g in games for c in g.choice]
    msgs = sum(len(g.transcript) for g in games)
    turns = [u for g in games for u in g.transcript] + [d for g in games for d in g.decision_info]
    thinking, forced = [t["thinking"] for t in turns], [t["forced"] for t in turns]
    return {
        **{k: oc[k] / n for k in OUTCOMES},
        "one_gives": (oc["GIVE/TAKE"] + oc["TAKE/GIVE"]) / n,
        "give_rate": choices.count(GIVE) / len(choices),
        "mean_pooled_payoff": sum(sum(g.payoffs()) for g in games) / n,
        "mean_messages": msgs / n,
        "ended_early": sum(g.ended_early for g in games) / n,
        "truncated_msgs": sum(u["truncated"] for g in games for u in g.transcript) / max(1, msgs),
        # reasoning: words of thinking per turn, and how often </think> had to be inserted (talk + decision turns)
        "think_words": sum(len(t.split()) for t in thinking) / max(1, len(thinking)),
        "forced_close": sum(forced) / max(1, len(forced)),
    }


def print_outcomes(label, games):
    oc = Counter(g.outcome() for g in games)
    st = game_stats(games)
    print(f"{label}: {len(games)} games (outcome = first speaker / second speaker)")
    for k in OUTCOMES:
        print(f"  {k:<10} {oc[k]:>4}  {oc[k] / len(games):6.1%}")
    print(f"  give rate {st['give_rate']:.1%} | mean pooled payoff {st['mean_pooled_payoff']:.2f} | "
          f"messages/game {st['mean_messages']:.1f} | ended early {st['ended_early']:.0%} | "
          f"cut-off messages {st['truncated_msgs']:.0%}")
    print(f"  thinking words/turn {st['think_words']:.0f} | </think> inserted (budget forcing) {st['forced_close']:.0%}")


def show_game(g, i=None, thinking=True):
    """Full trajectory of both agents: every turn in order (private reasoning, then the message the partner saw),
    then each agent's full decision reply."""
    name = {g.first: "A (first)", 1 - g.first: "B (second)"}
    print(f"{'-' * 30} game {'' if i is None else i}: {g.outcome()}, payoffs A/B "
          f"{g.payoffs()[g.first]}/{g.payoffs()[1 - g.first]}{', ended early' if g.ended_early else ''} {'-' * 30}")
    def think(t):
        if thinking and t["thinking"]:
            print(f"  (private reasoning{', budget-forced' if t['forced'] else ''})\n  " + t["thinking"].replace("\n", "\n  "))
    for u in g.transcript:
        print(f"[{name[u['speaker']]}{' - CUT OFF, not delivered' if u['truncated'] else ''}]")
        think(u)
        print(f"  (message)\n{u['text']}\n")
    for p in (g.first, 1 - g.first):
        print(f"[{name[p]} decides -> {g.choice[p] or 'INVALID'}]")
        think(g.decision_info[p])
        print(f"  (reply)\n{g.decision_text[p]}\n")


def show_raw(g, pid):
    """The exact token sequence one agent saw and produced (system prompt omitted)."""
    text = tok.decode(g.seats[pid].tokens)
    print(text[text.index(f"{IM_START}user"):])


def evaluate(label, policy):
    games = play_games(cfg.eval_games, (policy, policy))
    with open(os.path.join(cfg.log_path, f"eval_{policy}.jsonl"), "w") as f:
        for i, g in enumerate(games):
            f.write(json.dumps(g.record(condition=label, game=i)) + "\n")
    print_outcomes(label, games)
    print()
    for i, g in enumerate(games[:cfg.show_games]):
        show_game(g, i)
    return games


def train(adapter, pooled):
    history = []
    n_games = cfg.groups_per_batch * cfg.games_per_group
    print(f"adapter '{adapter}': {n_games} games/step, {cfg.steps} steps, reward={'pooled' if pooled else 'own payoff'}")
    print(f"{'step':>4} {'GG':>5} {'GT':>5} {'TG':>5} {'TT':>5} {'INV':>5} {'give':>5} {'pooled':>6} {'think':>5} {'forced':>6} {'gnorm':>6} {'sec':>5}")
    with open(os.path.join(cfg.log_path, f"train_{adapter}.jsonl"), "w") as f:
        for step in range(cfg.steps):
            t0 = time.time()
            games = play_games(n_games, (adapter, adapter), pooled, first_offset=step)
            st = {"step": step, **game_stats(games), **train_step(games, adapter), "sec": time.time() - t0}
            history.append(st)
            for i, g in enumerate(games):
                f.write(json.dumps(g.record(step=step, game=i)) + "\n")
            print(f"{step:>4} " + " ".join(f"{st[k]:>5.2f}" for k in OUTCOMES) +
                  f" {st['give_rate']:>5.2f} {st['mean_pooled_payoff']:>6.2f} {st['think_words']:>5.0f} {st['forced_close']:>6.2f}"
                  f" {st['grad_norm']:>6.3f} {st['sec']:>5.0f}")
    model.save_pretrained(os.path.join(cfg.log_path, "adapters"), selected_adapters=[adapter])
    return history

## (i) Base model

In [ ]:
results = {"base model": evaluate("base model", "base")}

In [ ]:
# The exact context of both agents in game 0 (everything each one saw and said, minus the system prompt).
for pid in (results["base model"][0].first, 1 - results["base model"][0].first):
    print(f"{'=' * 40} agent {'A (first)' if pid == results['base model'][0].first else 'B (second)'} {'=' * 40}")
    show_raw(results["base model"][0], pid)

## (ii) Pooled-reward RL

In [ ]:
histories = {"pooled": train("pooled", pooled=True)}

In [ ]:
results["pooled-reward RL"] = evaluate("pooled-reward RL", "pooled")

## (iii) Self-reward RL (control)

In [ ]:
histories["selfish"] = train("selfish", pooled=False)

In [ ]:
results["self-reward RL"] = evaluate("self-reward RL", "selfish")

## Summary

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

summary = pd.DataFrame({label: game_stats(gs) for label, gs in results.items()}).T
counts = pd.DataFrame({label: Counter(g.outcome() for g in gs) for label, gs in results.items()}).T.reindex(columns=OUTCOMES).fillna(0).astype(int)
print("Outcome counts (first speaker / second speaker):")
display(counts)
print("Fractions and other statistics:")
display(summary.style.format("{:.2f}"))
summary.join(counts, rsuffix="_count").to_csv(os.path.join(cfg.log_path, "summary.csv"))

series = [("GIVE/GIVE", "Give/Give", "#1baf7a"), ("one_gives", "one gives, one takes", "#2a78d6"),
          ("TAKE/TAKE", "Take/Take", "#eb6834"), ("INVALID", "invalid", "#eda100")]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, title) in zip(axes, [("pooled", "Pooled-reward RL"), ("selfish", "Self-reward RL")]):
    steps = [h["step"] for h in histories[name]]
    for key, label, color in series:
        ax.plot(steps, [h[key] for h in histories[name]], color=color, lw=2, label=label)
    ax.set(title=f"{title}: outcomes per step", xlabel="step", ylim=(0, 1))
    ax.grid(alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("fraction of games")
axes[0].legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Optional: keep the run (adapters, every training/eval game as JSONL, summary.csv).
# from google.colab import drive; drive.mount("/content/drive")
# !cp -r {cfg.log_path} /content/drive/MyDrive/